In [5]:
import requests
import pandas as pd
import time
import os
import json

In [3]:
# Download Data
# the results will be stored in the folder "data"
# the folder structure should be:
# project-chaggg/
# ├── data/
# │   └── chicago_crime_data/
# │       ├── batches/
# │       └── chicago_crimes_2001_2025.csv

# ── Config ────────────────────────────────────────────────────────────────────
APP_TOKEN    = "YOUR APP TOKEN HERE"
BASE_URL     = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
BATCH_SIZE   = 50000
BASE_DIR = os.getcwd()
OUTPUT_DIR   = os.path.join(BASE_DIR, "data", "chicago_crime_data")
FINAL_OUTPUT = os.path.join(OUTPUT_DIR, "chicago_crimes_2001_2025.csv")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
BATCH_DIR = os.path.join(OUTPUT_DIR, "batches") #this line was not in the last version
os.makedirs(BATCH_DIR, exist_ok=True)

EXPECTED_COLUMNS = [
    'id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
    'description', 'location_description', 'arrest', 'domestic', 'beat',
    'district', 'ward', 'community_area', 'fbi_code', 'year', 'updated_on',
    'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 'location'
]

# ── Progress helpers ──────────────────────────────────────────────────────────
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"offset": 0, "batch_num": 1, "total_records": 0}

def save_progress(offset, batch_num, total):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"offset": offset, "batch_num": batch_num, "total_records": total}, f)

# ── Fetch ─────────────────────────────────────────────────────────────────────
def fetch_batch(offset, retries=3):
    params = {
        "$limit":      BATCH_SIZE,
        "$offset":     offset,
        "$order":      "date ASC",
        "$$app_token": APP_TOKEN
    }
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(BASE_URL, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  [Attempt {attempt}/{retries}] Failed: {e}")
            if attempt < retries:
                time.sleep(5 * attempt)
            else:
                raise

# ── Align columns ─────────────────────────────────────────────────────────────
def align_columns(df):
    for col in EXPECTED_COLUMNS:
        if col not in df.columns:
            df[col] = None
    return df[EXPECTED_COLUMNS]

# ── Download ──────────────────────────────────────────────────────────────────
def download_all():
    if os.path.exists(FINAL_OUTPUT):
        print(f"File already exists: {FINAL_OUTPUT}\nDownload skipped.")
        return
    progress  = load_progress()
    offset    = progress["offset"]
    batch_num = progress["batch_num"]
    total     = progress["total_records"]

    print(f"{'Resuming' if offset > 0 else 'Starting'} download "
          f"(batch={batch_num}, offset={offset}, saved={total})\n")

    while True:
        batch_file = os.path.join(BATCH_DIR, f"batch_{batch_num:04d}.csv")

        if os.path.exists(batch_file):
            print(f"Batch {batch_num} already exists, skipping...")
            offset    += BATCH_SIZE
            batch_num += 1
            continue

        print(f"Fetching batch {batch_num} (offset={offset})...")
        batch = fetch_batch(offset)

        if not batch:
            print("No more data. Download complete.")
            break

        df = align_columns(pd.DataFrame(batch))
        df.to_csv(batch_file, index=False)

        total     += len(batch)
        offset    += BATCH_SIZE
        batch_num += 1
        save_progress(offset, batch_num, total)
        print(f"  -> {len(batch)} records. Total: {total}")
        time.sleep(0.5)

    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)

# ── Stream-merge batches ──────────────────────────────────────────────────────
def combine_batches():
    if os.path.exists(FINAL_OUTPUT):
        print(f"Final file already exists: {FINAL_OUTPUT}, skipping merge.")
        return

    batch_files = sorted(
        os.path.join(BATCH_DIR, f)
        for f in os.listdir(BATCH_DIR) if f.endswith(".csv")
    )
    print(f"\nMerging {len(batch_files)} batches into final CSV...")

    with open(FINAL_OUTPUT, "w", encoding="utf-8") as out:
        for i, path in enumerate(batch_files):
            df = pd.read_csv(path, low_memory=False)
            df = align_columns(df)                      # re-align just in case
            df.to_csv(out, index=False, header=(i == 0))
            print(f"  Merged {path} ({len(df)} rows)")

    print(f"\nDone! Saved to: {FINAL_OUTPUT}")

# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    download_all()
    combine_batches()

File already exists: c:\Users\andaa\OneDrive - Hertie School\MPP\4rd Semester\Algorithms\project-chaggg\notebooks\data\chicago_crime_data\chicago_crimes_2001_2025.csv
Download skipped.
Final file already exists: c:\Users\andaa\OneDrive - Hertie School\MPP\4rd Semester\Algorithms\project-chaggg\notebooks\data\chicago_crime_data\chicago_crimes_2001_2025.csv, skipping merge.


In [9]:
# Load the data

df = pd.read_csv("data/chicago_crime_data/chicago_crimes_2001_2025.csv", low_memory = False)

In [10]:
# Check the basic info of data

print(df.describe())
print(df.shape)          # rows and columns
print(df.dtypes)         # data types
print(df.columns.tolist()) # column names

                 id          beat      district          ward  community_area  \
count  8.476070e+06  8.476070e+06  8.476023e+06  7.861252e+06    7.862385e+06   
mean   7.556887e+06  1.183327e+03  1.129599e+01  2.278741e+01    3.737305e+01   
std    3.804702e+06  7.037585e+02  6.964312e+00  1.385922e+01    2.154745e+01   
min    6.340000e+02  1.110000e+02  1.000000e+00  1.000000e+00    0.000000e+00   
25%    4.088515e+06  6.210000e+02  6.000000e+00  1.000000e+01    2.300000e+01   
50%    7.554550e+06  1.034000e+03  1.000000e+01  2.300000e+01    3.200000e+01   
75%    1.096821e+07  1.731000e+03  1.700000e+01  3.400000e+01    5.600000e+01   
max    1.412427e+07  2.535000e+03  3.100000e+01  5.000000e+01    7.700000e+01   

               year  x_coordinate  y_coordinate      latitude     longitude  
count  8.476070e+06  8.381409e+06  8.381409e+06  8.381409e+06  8.381409e+06  
mean   2.011093e+03  1.164663e+06  1.885920e+06  4.184256e+01 -8.767126e+01  
std    7.136615e+00  1.694813e+04  3

In [13]:
df.columns.tolist()

['id',
 'case_number',
 'date',
 'block',
 'iucr',
 'primary_type',
 'description',
 'location_description',
 'arrest',
 'domestic',
 'beat',
 'district',
 'ward',
 'community_area',
 'fbi_code',
 'year',
 'updated_on',
 'x_coordinate',
 'y_coordinate',
 'latitude',
 'longitude',
 'location']

In [14]:
# Trabajando sobre el df que YA tienes en memoria
df_2001_2025 = df[df['year'].between(2001, 2025)]

In [11]:
# Filter the data from 2001 to 2025

df = pd.read_csv(FINAL_OUTPUT, low_memory=False)
df = df[df['year'].between(2001, 2025)]
df.to_csv(FINAL_OUTPUT, index=False)
print(df['year'].value_counts().sort_index())
print(f"\nTotal records (2001-2025): {len(df)}")

NameError: name 'FINAL_OUTPUT' is not defined

In [15]:
df_2001_2025.to_parquet("data/chicago_crime_data/chicago_2001_2025.parquet", index=False)
# o CSV
# df_2001_2025.to_csv("data/chicago_crime_data/chicago_2001_2025.csv", index=False)

In [1]:
# Check missing data

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

missing_summary

NameError: name 'df' is not defined

## Column Alignment Check

All 22 columns are present and correctly ordered — no misalignment detected.

Missing values are due to **genuine data gaps**, not structural issues:

| Column | Missing | Reason |
|--------|---------|--------|
| `ward` / `community_area` | ~614k | Early records do not include these fields |
| `x/y_coordinate` / `latitude` / `longitude` / `location` | ~94k | Addresses that could not be geocoded |
| `location_description` | ~15k | Small number of records with no venue description |
| `district` | 47 | Negligible |